In [28]:
# Accessing Google API key 
import os
from kaggle_secrets import UserSecretsClient

try:
    Capstone_project_key = UserSecretsClient().get_secret("Capstone_project_key")
    os.environ["Capstone_project_key"] = Capstone_project_key
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'Capstone_project_key' to your Kaggle secrets. Details: {e}"
    )

✅ Gemini API key setup complete.


In [29]:
# Accessing Food API key: https://fdc.nal.usda.gov/api-guide
import os
from kaggle_secrets import UserSecretsClient

try:
    Food_API = UserSecretsClient().get_secret("Food_API")
    os.environ["Food_API"] = Food_API
    print("✅ Food API key setup complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'Capstone_project_key' to your Kaggle secrets. Details: {e}"
    )

✅ Food API key setup complete.


In [30]:
#importing ADK components
from google.adk.agents import Agent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types


print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


In [31]:
#retry config to feed into Agent parameters
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1, # Initial delay before first retry (in seconds)
    http_status_codes=[429, 500, 503, 504] # Retry on these HTTP errors
)

In [ ]:
#AlertAgent

In [ ]:
#GlucosePrediction

In [ ]:
#InsulinAgent

In [32]:
# Tool: Food API
# Food with <= max_carbs can be searched using this tool
API_KEY = Food_API
def search_food_by_carbs(food_name: str, max_carbs: float):
    url = "https://api.nal.usda.gov/fdc/v1/foods/search"

    params = {
        "query": food_name,
        "pageSize": 20,
        "api_key": API_KEY
    }

    r = requests.get(url, params=params).json()

    foods = []

    for food in r["foods"]:
        nutrients = food.get("foodNutrients", [])
        nutrient_map = {n["nutrientName"]: n["value"] for n in nutrients}

        carbs = nutrient_map.get("Carbohydrate, by difference")
        protein = nutrient_map.get("Protein")
        calories = nutrient_map.get("Energy (kcal)") or nutrient_map.get("Energy")

        calories_from_carbs = carbs * 4 if carbs is not None else None

        if carbs is not None and carbs <= max_carbs:
            foods.append({
                "name": food["description"],
                "carbs_g": carbs,
                "protein_g": protein,
                "calories_kcal": calories,
                "calories_from_carbs": calories_from_carbs,
                "serving_size": food.get("servingSize"),
                "serving_unit": food.get("servingSizeUnit")
            })

    return foods

In [27]:
# testing
search_food_by_carbs("",20)

[{'name': 'Abalone',
  'carbs_g': 7.46,
  'protein_g': 21.24,
  'calories_kcal': 177,
  'calories_from_carbs': 29.84,
  'serving_size': None,
  'serving_unit': None},
 {'name': 'Abiyuch, raw',
  'carbs_g': 17.6,
  'protein_g': 1.5,
  'calories_kcal': 69.0,
  'calories_from_carbs': 70.4,
  'serving_size': None,
  'serving_unit': None},
 {'name': 'Acerola juice, raw',
  'carbs_g': 4.8,
  'protein_g': 0.4,
  'calories_kcal': 23.0,
  'calories_from_carbs': 19.2,
  'serving_size': None,
  'serving_unit': None},
 {'name': 'Acerola, (west indian cherry), raw',
  'carbs_g': 7.69,
  'protein_g': 0.4,
  'calories_kcal': 32.0,
  'calories_from_carbs': 30.76,
  'serving_size': None,
  'serving_unit': None},
 {'name': 'Acorn stew (Apache)',
  'carbs_g': 9.22,
  'protein_g': 6.81,
  'calories_kcal': 95.0,
  'calories_from_carbs': 36.88,
  'serving_size': None,
  'serving_unit': None},
 {'name': 'Adobo, with noodles',
  'carbs_g': 8.14,
  'protein_g': 16.89,
  'calories_kcal': 172,
  'calories_from_c

In [33]:
#MealAgent
MealAgent = Agent(
    name="MealRecommenderAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # This instruction tells the Meal Agent HOW to use its tools (which are the other agents).
    instruction="""

Role

You are a Diabetes Nutrition Coach Agent.
Your goal is to recommend meals and hydration strategies that help keep the user's blood glucose within 90–150 mg/dL.

Use food nutrition data from the USDA FoodData Central API to construct meal recommendations.Use search_food_by_carbs tool
to interact with the API.

Decision Rules
1. Hypoglycemia Management

If glucose < 70 mg/dL:

Recommend 15 g fast-acting carbohydrates (e.g., fruit juice or glucose tablets).

Ask the user to wait 15 minutes.

Recommend rechecking glucose levels.

If glucose remains below 70 mg/dL, repeat the 15 g carbohydrate treatment.

After glucose rises above 70 mg/dL, recommend a balanced meal.

2. Hyperglycemia Hydration

If glucose > 180 mg/dL:

Recommend drinking 500 mL–1 L of water to support hydration.

3. Meal Construction Rules

When recommending meals, present food items in the following order:

Protein

Vegetables

Carbohydrates

Meals should:

Include lean protein

Include non-starchy vegetables

Include controlled carbohydrate portions

4. Carbohydrate Impact Assumption

Assume:

10 g of carbohydrates may increase blood glucose by approximately 30–50 mg/dL.

Use this assumption to estimate appropriate carbohydrate quantities for the desired change in the blood glucose. 

5. Medication Timing Rule

If the user has recently taken insulin or oral diabetes medication, recommend meals 15 minutes after medication intake when appropriate.

Tool Usage

To generate food recommendations:

use search_food_by_carbs tool.

The tool should retrieve:

food name

macronutrients

carbohydrate content

serving size

Select foods that satisfy the meal rules.

Output Format

Provide recommendations in the following format:

Glucose Status:
(normal / high / low)

Hydration Recommendation:
(if applicable)

Meal Recommendation

Protein:

example food + portion

Vegetables:

example food + portion

Carbohydrates:

example food + portion

Estimated Carbohydrates:
XX grams

Estimated impact of glucose level:
This meal provides an expected XX mg/dL change to bring blood glucose within the target range of 90–150 mg/dL. 

Additional Guidance:
(e.g., recheck glucose after 15 minutes)
"""
, tools = [search_food_by_carbs]
)
print("✅ MealAgent created.")

✅ MealAgent created.


In [ ]:
#ExerciseAgent

In [ ]:
#SafetyGuardAgent

In [34]:
## Creating a dummy prediction model
# input: <min_past and max_past that automatically create random 24 past CGM data points within min & max range
# output: <min_pred, max_pred and randomly generated 12 future CGM data points within min and max range

import random

class DummyCGMModel:

    def predict(self, min_past, max_past):

        # Generate 24 past CGM values
        past_cgm = [random.randint(min_past, max_past) for _ in range(24)]

        # Define prediction range
        min_pred = min(past_cgm) - random.randint(-20, 30)
        max_pred = max(past_cgm) + random.randint(-20, 80)

        # Generate 12 future predictions
        future_cgm = [random.randint(min_pred, max_pred) for _ in range(12)]

        return {
            "past_cgm_24_points": past_cgm,
            "min_pred": min_pred,
            "max_pred": max_pred,
            "future_cgm_12_points": future_cgm
        }

# Save the model with joblib
import joblib

model = DummyCGMModel()

joblib.dump(model, "dummy_cgm_model.joblib")

print("Model saved!")



Model saved!


In [35]:
# prediction tool for the Main agent
def predict_glucose(cgm_values):
    model = joblib.load("dummy_cgm_model.joblib")
    prediction = model.predict(min_past, max_past) # min max of past 24 values
    return prediction.tolist()

In [4]:
# Root Coordinator: Orchestrates the workflow by calling the sub-agents as tools.
Main_agent = Agent(
    name="Orchestrator Agent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # This instruction tells the root agent HOW to use its tools (which are the other agents).
    instruction="""
    System Role

You are a Blood Glucose Coaching Orchestrator Agent that helps users with diabetes maintain their blood glucose within the target range of 90–150 mg/dL.

Your job is to coordinate multiple specialized agents and tools to generate safe, personalized recommendations.

You do not guess medical information. Instead, you call the appropriate tools and synthesize their results into clear recommendations.

Primary Objective

Maintain the user’s glucose within 90–150 mg/dL by coordinating:

glucose prediction using glucose prediction tool

medication reminders

insulin dosing guidance

meal recommendations

exercise recommendations

hydration advice

Workflow (You MUST follow this order)
Step 1 — Medication Alert

Check whether the user has scheduled medication.

If the current time matches a medication schedule:

Call the AlertAgent tool to notify the user about:

oral medication

insulin

GLP-1 agonist

Step 2 — Predict Future Glucose

You must call the Glucose Prediction tool.

Inputs:

last 2 hours of glucose readings

readings at 5-minute intervals

Output:

predicted glucose for next 1 hour

predictions at 5-minute intervals

Use this prediction to guide all subsequent decisions.

Step 3 — Insulin Recommendation

If the user takes short-acting insulin:

Call the InsulinAgent tool.

Use:

current glucose OR

predicted glucose at the time of the meal.

Return the recommended insulin dosage.

Step 4 — Meal Recommendation

If the current time corresponds to the user’s preferred meal time (breakfast, lunch, or dinner):

Call the MealAgent tool.

Provide:

predicted glucose trajectory

user dietary preferences

glucose target range

The meal recommendation should help keep predicted glucose within 90–150 mg/dL.

Include hydration guidance if appropriate.

Step 5 — Exercise Recommendation

Call the ExerciseAgent tool to generate an exercise recommendation.

Use:

predicted glucose trend

time since last meal

safety constraints

Exercise recommendations should help maintain glucose within the target range.

Safety Rules (Always Follow)

Avoid recommending exercise if glucose is < 70 mg/dL.

Hypoglycemia protocol

If glucose < 70 mg/dL:

Eat fast-acting carbohydrates first

Wait 15 minutes

Recheck glucose

Repeat if still low

Medication timing

Pre-meal medication should be taken 15 minutes before the meal.

Exercise timing

Post-meal exercise should typically occur ~2 hours after eating.

Tool Usage Rules

You must follow these rules:

Do not fabricate glucose predictions.

Do not fabricate insulin dosage.

Do not invent nutritional values.

Always use the appropriate tool when required.

Tools available:

AlertAgent

predict_glucose

InsulinAgent

MealAgent

ExerciseAgent

Final Response Format

After completing all tool calls, present a clear summary to the user containing:

1. Glucose Outlook

Brief description of predicted glucose trend.

2. Medication Reminder

If applicable.

3. Meal Recommendation

Suggested meal and hydration.

4. Exercise Recommendation

Activity type and timing.

5. Safety Notes

Any warnings about hypoglycemia or high glucose.

Keep explanations simple, supportive, and actionable.
    """,
    # We wrap the sub-agents in `AgentTool` to make them callable tools for the root agent.
    tools=[predict_glucose], 
    sub_agents=[AlertAgent,InsulinAgent,MealAgent,ExerciseAgent],
)

print("✅ root_agent created.")

NameError: name 'AlertAgent' is not defined

In [ ]:
#Runner to run the Main Agent
runner = InMemoryRunner(agent=Main_agent)

print("✅ Runner created.")

In [ ]:
# Provide user input to execute the main agent response
# Execute the agent 
response = await runner.run_debug(
"""
min_past = minimum of CGM reading past 2 hours in 5 min interval =60
max_past = maximum of CGM reading past 2 hours in 5 min interval = 200
Weight	=[75]		# in KG	
Height	=[1.65]		# in meter	
Diet Preference = [Western]				
				
Usual meal time	= [breakfast: 8am , lunch: 12:30PM, dinner: 7PM ]	#breakfast, lunch, Dinner timings		
				
Oral Medications dosing	= [pre-meal] #pre-meal/once a day/once a week. What time?				
			
insulin intake =[Y] #YN			
GLP-1 Agonist intake = [N] # YN->when
"""

)